# MCP 클라이언트를 위한 AgentCore Gateway 및 Okta 기반의 안전한 Authorization Code 흐름 구축

이 Notebook에서는 **Okta**를 Identity Provider로 사용하여 블로그 게시물 **"MCP 클라이언트와 AgentCore Gateway를 사용한 안전한 Authorization Code 흐름 설정"**의 구현 과정을 살펴봅니다.

이 Notebook을 완료하면 다음을 구성하게 됩니다.
- PKCE를 사용하는 Authorization Code 흐름으로 구성된 Okta OIDC 애플리케이션
- Okta를 가리키는 JWT 인바운드 인증이 설정된 AgentCore Gateway
- Gateway 도구에 액세스하기 전에 Okta를 통해 인증하는 Kiro IDE 구성

### 사전 요구 사항
- AgentCore Gateway 권한이 있는 AWS 계정
- Okta 개발자 계정(https://developer.okta.com/signup/)
- `mcp-remote` 설치(`npm install -g mcp-remote`)
- 로컬 환경에 Kiro IDE 설치
- 이 Python 환경에 `boto3` 설치

## 1단계: Okta를 Identity Provider로 구성

Okta 계정이 이미 있다면 로그인합니다. 계정이 없다면 https://developer.okta.com/signup/으로 이동하여 **"Sign up for Integrator Free Plan"**을 선택합니다.

### 1.1 테스트 사용자 생성

1. Okta 관리 콘솔에 로그인합니다.
2. **Directory** > **People**을 선택하고 **Add person**을 클릭합니다.

![사용자 추가](assets/images/addPerson.png)

3. 양식을 작성합니다.
   - **Activation**에서 **Activate now**를 선택합니다.
   - **I will set password**를 선택하고 암호를 설정합니다.
   - **User must change password on first login**을 선택 해제합니다.
   - **Save**를 클릭합니다.

![사용자 추가 양식](assets/images/addPersonForm.png)

### 1.2 OIDC 애플리케이션 생성

1. **Applications** > **Create App Integration**을 선택합니다.

![앱 통합 생성](assets/images/createAppIntegration.png)

2. 로그인 방식으로 **OIDC - OpenID Connect**를 선택합니다.
3. 애플리케이션 유형으로 **Single-Page Application**을 선택하고 **Next**를 클릭합니다.

![앱 통합 양식 - 로그인 방식](assets/images/appIntegrationForm1.png)

> **중요:** "Web Application"이 아닌 **Single-Page Application**을 선택해야 합니다. SPA 앱은 client secret 없이 PKCE를 사용하는 공개 OAuth 클라이언트이며, 이는 `mcp-remote`가 요구하는 방식과 정확히 일치합니다. "Web Application"(기밀 클라이언트)은 토큰 교환 중 `client_secret`이 필요하므로 인증 흐름이 실패합니다.

#### 애플리케이션 구성

   a. **App integration name:** `AgentCore Gateway Client`(또는 원하는 이름)을 입력합니다.
   
   b. **Proof of possession**은 선택하지 않은 상태로 둡니다.
   
   c. **Grant type:** **Authorization Code**와 **Refresh Token**을 선택합니다.
   
   d. **Sign-in redirect URI:** 다음과 같이 설정합니다.
   ```
   http://localhost:3334/oauth/callback
   ```
   이 URL은 Kiro IDE와의 OAuth 흐름 중 `mcp-remote`가 사용하는 콜백 URL입니다. 다른 기본 redirect URI는 모두 제거합니다.

![앱 통합 양식 - Grant 유형 및 Redirect URI](assets/images/appIntegrationForm2.png)
   
   e. **Sign-out redirect URI**는 그대로 둡니다.
   
   f. **Assignments**에서 **Allow everyone in your organization to access**를 선택하고 **Enable immediate access**가 선택된 상태로 둔 다음 **Save**를 클릭합니다.

![앱 통합 양식 - 할당](assets/images/appIntegrationForm3.png)
   
   g. 나중에 사용할 수 있도록 **Client ID**를 복사합니다.

> **참고:** 이 애플리케이션은 Single-Page Application(공개 클라이언트)이므로 Client Secret이 없습니다. 대신 이 흐름에서는 `mcp-remote`와 같은 데스크톱/CLI OAuth 클라이언트에 권장되는 방식인 PKCE(Proof Key for Code Exchange)를 사용합니다.

### 1.3 Authorization Server 구성

1. 왼쪽 메뉴에서 **Security** > **API**를 선택하고 authorization server의 이름(예: `default`)을 클릭합니다.

2. **Audience**를 복사하여 나중에 사용할 수 있도록 저장합니다.
   > **참고:** 다른 앱에 영향을 주지 않도록 audience를 변경할 계획이라면 새 authorization server를 생성하는 것이 좋습니다.

3. **Scopes** > **Add Scope**를 클릭합니다.
   - **Name:** `okta.myAccount.read`
   - **Display Phrase**와 **Description**을 입력합니다.
   - **User Consent**를 **implicit**으로 설정합니다.
   - 다른 설정은 기본값으로 둡니다.
   - **Save**를 클릭합니다.

![Scope 추가](assets/images/addScope.png)

4. **Claims**를 클릭하고 다음 claim을 추가합니다.
   - `client_id` claim

![Claim 추가 - Client ID](assets/images/addClaimClientId.png)

   - `scope` claim

![Claim 추가 - Scope](assets/images/addClaimScope.png)
   
   이 claim들은 AgentCore Gateway가 JWT 토큰의 `cid` custom claim을 검증하는 데 필요합니다.

5. **Access Policies** > **Add New Access Policy**를 클릭합니다.
   - **Name**과 **Description**을 입력하고 **Create Policy**를 클릭합니다.
   - **Add rule**을 클릭하고 **Rule Name**을 입력한 다음 **Create Rule**을 클릭합니다.

### 1.4 구성 값 저장

다음 단계에서 아래 값이 필요합니다.

| 값 | 확인 위치 | 예시 |
|-------|------------------|---------|
| **Client ID** | Application > General 탭 | `0oaz7147z771FZmdQ697` |
| **Audience** | Security > API > Authorization Server | `api://default` |
| **Okta Domain** | 관리 콘솔 오른쪽 상단 또는 Settings | `dev-12345678.okta.com` |
| **Discovery URL** | 도메인에서 파생 | `https://{domain}/oauth2/default/.well-known/openid-configuration` |

## 2단계: Okta JWT 인증을 사용하는 AgentCore Gateway 생성

Okta authorization server를 가리키는 JWT 기반 인바운드 인증을 사용하여 새 AgentCore Gateway를 생성합니다.

> **참고:** Gateway의 authorizer 유형은 생성 후 변경할 수 없으므로 처음부터 `CUSTOM_JWT`로 구성합니다.

아래에 Okta 구성 값을 입력하고 셀을 실행합니다.

In [2]:
# --- Okta 값 입력 ---
OKTA_DOMAIN = input("Enter your Okta domain (e.g., dev-12345678.okta.com): ")
OKTA_CLIENT_ID = input("Enter your Okta Client ID: ")
OKTA_AUDIENCE = input("Enter your Okta Audience: ")

# --- Gateway 값 ---
GATEWAY_NAME = "okta-auth-code-gateway"
GATEWAY_ROLE_ARN = "arn:aws:iam::265666655061:role/AgentCoreGatewayExecutionRole"
REGION = "us-west-2"

# 파생 값
DISCOVERY_URL = f"https://{OKTA_DOMAIN}/oauth2/default/.well-known/openid-configuration"

print(f"\nDiscovery URL: {DISCOVERY_URL}")
print(f"Gateway Name:  {GATEWAY_NAME}")


Discovery URL: https://trial-6312662.okta.com/oauth2/default/.well-known/openid-configuration
Gateway Name:  okta-auth-code-gateway


In [3]:
import boto3

client = boto3.client("bedrock-agentcore-control", region_name=REGION)

response = client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": DISCOVERY_URL,
            "allowedAudience": [OKTA_AUDIENCE],
            "allowedClients": [OKTA_CLIENT_ID],
            "customClaims": [
                {
                    "inboundTokenClaimName": "cid",
                    "inboundTokenClaimValueType": "STRING",
                    "authorizingClaimMatchValue": {
                        "claimMatchValue": {"matchValueString": OKTA_CLIENT_ID},
                        "claimMatchOperator": "EQUALS",
                    },
                }
            ],
        }
    },
)

GATEWAY_ID = response["gatewayId"]
GATEWAY_URL = response["gatewayUrl"]

print(f"Gateway ID:  {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")
print(f"Status:      {response['status']}")
print(f"Auth Type:   {response['authorizerType']}")
print("\nSave the Gateway ID and URL for the next steps.")

Gateway ID:  okta-auth-code-gateway-bvgdzbk29p
Gateway URL: https://okta-auth-code-gateway-bvgdzbk29p.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp
Status:      CREATING
Auth Type:   CUSTOM_JWT

Save the Gateway ID and URL for the next steps.


## 3단계: Gateway에 인증이 필요한지 확인

인증되지 않은 MCP 요청을 전송하여 Gateway에 유효한 JWT 토큰이 필요한지 확인합니다.

> **참고:** `GATEWAY_URL`은 이전 셀에서 생성 응답을 바탕으로 설정되었습니다. 먼저 2단계를 실행했는지 확인합니다.

In [5]:
import requests
import json

GATEWAY_URL = f"https://{GATEWAY_ID}.gateway.bedrock-agentcore.{REGION}.amazonaws.com/mcp"

print(f"Testing: {GATEWAY_URL}\n")

resp = requests.post(
    GATEWAY_URL,
    headers={"Content-Type": "application/json"},
    json={"jsonrpc": "2.0", "method": "initialize", "params": {}, "id": 1},
)

print(f"Status Code: {resp.status_code}")
print("Headers:")
for k, v in resp.headers.items():
    if "auth" in k.lower() or "www" in k.lower():
        print(f"  {k}: {v}")
print(f"\nBody: {resp.text[:500]}")

if resp.status_code == 401:
    print("\n✓ Gateway correctly requires authentication.")
else:
    print(f"\n⚠ Expected 401, got {resp.status_code}. Check your Gateway configuration.")

Testing: https://okta-auth-code-gateway-bvgdzbk29p.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp

Status Code: 401
Headers:
  WWW-Authenticate: Bearer resource_metadata="https://okta-auth-code-gateway-bvgdzbk29p.gateway.bedrock-agentcore.us-west-2.amazonaws.com/.well-known/oauth-protected-resource"

Body: {"jsonrpc":"2.0","id":0,"error":{"code":-32001,"message":"Missing Bearer token"}}

✓ Gateway correctly requires authentication.


## 4단계: mcp-remote 설치

`mcp-remote`는 Kiro IDE와 Gateway 사이에서 OAuth 프록시 역할을 합니다. PKCE를 사용하는 Authorization Code 흐름을 처리하고, 토큰 갱신을 관리하며, 인증된 요청을 전달합니다.

Notebook이 아닌 터미널에서 다음 명령을 실행합니다.

```bash
npm install -g mcp-remote
```

## 5단계: Kiro IDE 구성

Kiro MCP 구성에 Gateway를 추가합니다. 아래 셀에서 필요한 JSON을 정확히 생성합니다.

> **Kiro IDE 관련 중요 사항:** Kiro는 셸의 `PATH` 또는 `HOME`을 상속하지 않고 MCP 서버 프로세스를 생성합니다. `mcp-remote` 바이너리의 전체 절대 경로를 지정하고 `env` 섹션에 `HOME`과 `PATH`를 명시적으로 설정해야 합니다. 전체 경로를 확인하려면 터미널에서 `which mcp-remote`를 실행합니다.

In [6]:
import shutil
import os

MCP_PORT = "3334"

# mcp-remote의 전체 경로 확인(Kiro IDE에 필요)
mcp_remote_path = shutil.which("mcp-remote") or "mcp-remote"
home_dir = os.path.expanduser("~")
path_env = os.path.dirname(mcp_remote_path) + ":/usr/local/bin:/usr/bin:/bin"

kiro_config = {
    "mcpServers": {
        "gateway-tools": {
            "command": mcp_remote_path,
            "args": [
                GATEWAY_URL,
                MCP_PORT,
                "--static-oauth-client-info",
                json.dumps(
                    {
                        "client_id": OKTA_CLIENT_ID,
                        "redirect_uris": [f"http://localhost:{MCP_PORT}/oauth/callback"],
                        "scope": "openid profile email offline_access",
                    }
                ),
            ],
            "env": {"PATH": path_env, "HOME": home_dir},
        }
    }
}

config_json = json.dumps(kiro_config, indent=2)
print("Add the following to ~/.kiro/settings/mcp.json:\n")
print(config_json)

Add the following to ~/.kiro/settings/mcp.json:

{
  "mcpServers": {
    "gateway-tools": {
      "command": "/Users/dssouto/.nvm/versions/node/v23.9.0/bin/mcp-remote",
      "args": [
        "https://okta-auth-code-gateway-bvgdzbk29p.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp",
        "3334",
        "--static-oauth-client-info",
        "{\"client_id\": \"0oa114sk6r11FF3On698\", \"redirect_uris\": [\"http://localhost:3334/oauth/callback\"], \"scope\": \"openid profile email offline_access\"}"
      ],
      "env": {
        "PATH": "/Users/dssouto/.nvm/versions/node/v23.9.0/bin:/usr/local/bin:/usr/bin:/bin",
        "HOME": "/Users/dssouto"
      }
    }
  }
}


### 구성 적용

1. `~/.kiro/settings/mcp.json`을 열거나 생성합니다.
2. 위 출력의 `gateway-tools` 항목을 `mcpServers` 섹션에 추가합니다.
3. 파일을 저장합니다.

> **팁:** Kiro IDE가 이미 열려 있다면 구성 변경을 자동으로 감지하고 새 MCP 서버에 연결을 시도하므로 다시 시작할 필요가 없습니다.

## 6단계: 엔드 투 엔드 흐름 테스트

`mcp.json`을 저장하면 Kiro IDE가 새 구성을 자동으로 감지하고 Gateway에 연결하기 시작합니다.

1. 브라우저에 Okta 로그인 페이지가 열립니다.
2. 1단계에서 생성한 테스트 사용자로 인증합니다.
3. 브라우저에 "Authorization successful! You may close this window"가 표시되는지 확인합니다.
4. Kiro IDE로 돌아가 MCP 서버 상태를 확인합니다. `gateway-tools` 옆에 녹색 체크 표시가 나타나야 합니다.

연결에 성공하면 MCP Logs에 다음과 같이 표시됩니다.
```
[info] [gateway-tools] Successfully connected and synced tools and resources for MCP server
```

### 내부 동작 방식

```
Kiro IDE → mcp-remote → Gateway returns 401 with auth metadata
    → mcp-remote opens browser to Okta login
    → User authenticates with Okta
    → Okta redirects to http://localhost:3334/oauth/callback with auth code
    → mcp-remote exchanges code for access token (using PKCE)
    → mcp-remote sends tool request with Bearer token
    → Gateway validates JWT (signature, expiry, cid claim)
    → Gateway proxies request to MCP server
    → Response flows back to Kiro IDE
```